In [20]:
import pandas as pd
import os
from itertools import combinations


In [21]:
#crab_eating_macaque，rhesus_macaque，human，mice，pig
os.chdir("/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/Homologue_newversion/Cell_Atlas_of_Aqueous_Humor")
crab_eating_macaque_rhesus_macaque=pd.read_csv("/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/ENS_M2M/cross_species_homologue_genes/crab_eating_macaque_rhesus_macaque_ENS_M2M.txt",sep="\t",usecols=range(2))
rhesus_macaque_human=pd.read_csv("/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/ENS_M2M/cross_species_homologue_genes/rhesus_macaque_human_ENS_M2M.txt",sep="\t",usecols=range(2))
human_mice=pd.read_csv("/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/ENS_M2M/cross_species_homologue_genes/human_mice_ENS_M2M.txt",sep="\t",usecols=range(2))
mice_pig=pd.read_csv("/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/ENS_M2M/cross_species_homologue_genes/mice_pig_ENS_M2M.txt",sep="\t",usecols=range(2))

In [22]:
print(mice_pig.shape)
print(mice_pig.iloc[0:5,:])

(84961, 2)
  Gene name Pig gene name
0     mt-Tf           NaN
1   mt-Rnr1           NaN
2     mt-Tv           NaN
3   mt-Rnr2           NaN
4    mt-Tl1           NaN


In [23]:
crab_eating_macaque_rhesus_macaque = crab_eating_macaque_rhesus_macaque.dropna(subset=[crab_eating_macaque_rhesus_macaque.columns[0], crab_eating_macaque_rhesus_macaque.columns[1]])#rhesus_macaque_human human_mice
rhesus_macaque_human = rhesus_macaque_human.dropna(subset=[rhesus_macaque_human.columns[0], rhesus_macaque_human.columns[1]])
human_mice = human_mice.dropna(subset=[human_mice.columns[0], human_mice.columns[1]])
mice_pig = mice_pig.dropna(subset=[mice_pig.columns[0], mice_pig.columns[1]])

print(crab_eating_macaque_rhesus_macaque.shape)
print(rhesus_macaque_human.shape)
print(human_mice.shape)
print(mice_pig.shape)

(21074, 2)
(21142, 2)
(24685, 2)
(17252, 2)


In [24]:
print(crab_eating_macaque_rhesus_macaque.iloc[0:5,:])
print(rhesus_macaque_human.iloc[0:5,:])
print(human_mice.iloc[0:5,:])
print(mice_pig.iloc[0:5,:])

   Gene name Macaque gene name
22     PGBD2             PGBD2
23     PGBD2             PGBD2
25     Y_RNA             Y_RNA
27   TCEANC2           TCEANC2
28     MEX3A             MEX3A
  Gene name Human gene name
0     PGBD2           PGBD2
1        U6      RNU6-1205P
2    ZNF692          ZNF692
3    ZNF672          ZNF672
4   SH3BP5L         SH3BP5L
   Gene name Mouse gene name
5     MT-ND1          mt-Nd1
9     MT-ND2          mt-Nd2
15    MT-CO1          mt-Co1
18    MT-CO2          mt-Co2
20   MT-ATP8         mt-Atp8
   Gene name Pig gene name
5     mt-Nd1           ND1
9     mt-Nd2           ND2
15    mt-Co1          COX1
18    mt-Co2          COX2
20   mt-Atp8          ATP8


In [25]:
crab_eating_macaque_rhesus_macaque=crab_eating_macaque_rhesus_macaque.rename(columns={crab_eating_macaque_rhesus_macaque.columns[0]:"crab_eating_macaque gene name"})
crab_eating_macaque_rhesus_macaque=crab_eating_macaque_rhesus_macaque.rename(columns={crab_eating_macaque_rhesus_macaque.columns[1]:"rhesus_macaque gene name"})
rhesus_macaque_human=rhesus_macaque_human.rename(columns={rhesus_macaque_human.columns[0]:"rhesus_macaque gene name"})
rhesus_macaque_human=rhesus_macaque_human.rename(columns={rhesus_macaque_human.columns[1]:"human gene name"})
human_mice=human_mice.rename(columns={human_mice.columns[0]:"human gene name"})
human_mice=human_mice.rename(columns={human_mice.columns[1]:"mice gene name"})
mice_pig=mice_pig.rename(columns={mice_pig.columns[0]:"mice gene name"})
mice_pig=mice_pig.rename(columns={mice_pig.columns[1]:"pig gene name"})

In [26]:
print(crab_eating_macaque_rhesus_macaque.shape)
print(rhesus_macaque_human.shape)
print(human_mice.shape)
print(mice_pig.shape)

(21074, 2)
(21142, 2)
(24685, 2)
(17252, 2)


In [27]:
crab_eating_macaque_rhesus_macaque = crab_eating_macaque_rhesus_macaque.drop_duplicates()
rhesus_macaque_human = rhesus_macaque_human.drop_duplicates()
human_mice=human_mice.drop_duplicates()
mice_pig=mice_pig.drop_duplicates()

In [28]:
print(crab_eating_macaque_rhesus_macaque.shape)
print(rhesus_macaque_human.shape)
print(human_mice.shape)
print(mice_pig.shape)

(16899, 2)
(19800, 2)
(24419, 2)
(17104, 2)


In [29]:
def cap_by_any_four(df, cols5, k=6, order_by=None):
    """
    对于任意“四列相同”的分组，最多保留 k 行。
    cols5: 5 个列名组成的列表
    order_by: 可选，用这些列先排序以决定“保留哪几条”（默认按原始顺序）
    """
    out = df.copy()

    # 决定保留顺序：不指定就按原始行序
    if order_by:
        out = out.sort_values(by=order_by, kind="stable")
    else:
        out = out.reset_index(drop=False).rename(columns={"index": "__order__"})\
                 .sort_values("__order__", kind="stable")

    for subset in combinations(cols5, 4):   # 5C4 个组合
        cc = out.groupby(list(subset)).cumcount()
        out = out[cc < k]                   # 超过 k 的行丢弃

    # 清理辅助列
    if "__order__" in out.columns:
        out = out.drop(columns="__order__")
    return out

In [30]:
crab_eating_macaque_rhesus_macaque_human=pd.merge(crab_eating_macaque_rhesus_macaque,rhesus_macaque_human,on="rhesus_macaque gene name",how="outer")
crab_eating_macaque_rhesus_macaque_human_mice=pd.merge(crab_eating_macaque_rhesus_macaque_human,human_mice,on="human gene name",how="outer")
crab_eating_macaque_rhesus_macaque_human_mice_pig=pd.merge(crab_eating_macaque_rhesus_macaque_human_mice,mice_pig,on="mice gene name",how="outer")
crab_eating_macaque_rhesus_macaque_human_mice_pig = crab_eating_macaque_rhesus_macaque_human_mice_pig.dropna()
crab_eating_macaque_rhesus_macaque_human_mice_pig = crab_eating_macaque_rhesus_macaque_human_mice_pig.drop_duplicates(subset=["crab_eating_macaque gene name","rhesus_macaque gene name","human gene name","mice gene name","pig gene name"], keep='first')
print(crab_eating_macaque_rhesus_macaque_human_mice_pig.shape)
print(crab_eating_macaque_rhesus_macaque_human_mice_pig.iloc[0:5,:])

(241121, 5)
  crab_eating_macaque gene name rhesus_macaque gene name human gene name  \
0                      C17orf49              C16H17orf49        C17orf49   
1                       C2orf68               C13H2orf68         C2orf68   
2                       C4orf19                C5H4orf19         C4orf19   
3                       C4orf54                C5H4orf54         C4orf54   
4                      C11orf58              C14H11orf58        C11orf58   

  mice gene name pig gene name  
0  0610010K14Rik      C17orf49  
1  0610030E20Rik       C2orf68  
2  0610040J01Rik       C4orf19  
3  1110002E22Rik       C4orf54  
4  1110004F10Rik      C11orf58  


In [31]:
# cols5 = ["crab_eating_macaque gene name","rhesus_macaque gene name",
#          "human gene name","mice gene name","pig gene name"]

# df = crab_eating_macaque_rhesus_macaque_human_mice_pig
# df_capped = cap_by_any_four(df, cols5, k=6)  # 任意四列相同的组，最多 6 条

# print(df_capped.shape)
# print(df_capped.iloc[:5, :])


In [32]:
crab_eating_macaque_rhesus_macaque_human_mice_pig.to_csv("crab_eating_macaque_rhesus_macaque_human_mice_pig.csv",index=False)